# 01 — Inspect running sim

다른 터미널에서 sim 이 떠 있다고 가정합니다:

```bash
./scripts/host/run.sh sim
```

이 노트북은 같은 호스트 netns 안에서 ROS 2 Jazzy 노드를 띄워
`/joint_states` 와 `/sim/diagnostics` 를 잠깐 구독하고, 받은 데이터를
matplotlib 으로 그립니다.

**Sanity 체크 — 이 노트북이 sim 컨테이너와 같은 도메인을 보고 있는가**

- compose 의 `network_mode: host` + `ROS_DOMAIN_ID` 를 똑같이 공유하므로,
  `run.sh jupyter` 컨테이너에서 그냥 `rclpy` 로 구독하면 됩니다.
- DDS 가 RMW=cyclonedds 로 일치해야 하는데, 둘 다 compose 가 같은 값을
  주입하므로 별도 설정이 필요하지 않습니다.

In [ ]:
import os
print('ROS_DISTRO        =', os.environ.get('ROS_DISTRO'))
print('ROS_DOMAIN_ID     =', os.environ.get('ROS_DOMAIN_ID'))
print('RMW_IMPLEMENTATION=', os.environ.get('RMW_IMPLEMENTATION'))

## 1. 토픽 구독 — `/joint_states`, `/sim/diagnostics`

`COLLECT_SECONDS` 동안 메시지를 모은 뒤 노드를 종료합니다.

In [ ]:
import time
import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, ReliabilityPolicy, HistoryPolicy
from sensor_msgs.msg import JointState
from diagnostic_msgs.msg import DiagnosticArray

COLLECT_SECONDS = 5.0

qos = QoSProfile(
    reliability=ReliabilityPolicy.RELIABLE,
    history=HistoryPolicy.KEEP_LAST,
    depth=50,
)

class Sniffer(Node):
    def __init__(self):
        super().__init__('nb_sniffer')
        self.joint_msgs = []
        self.diag_msgs = []
        self.create_subscription(JointState, '/joint_states', self._on_js, qos)
        self.create_subscription(DiagnosticArray, '/sim/diagnostics', self._on_diag, qos)

    def _on_js(self, msg):
        t = msg.header.stamp.sec + msg.header.stamp.nanosec * 1e-9
        self.joint_msgs.append((t, list(msg.name), list(msg.position), list(msg.velocity)))

    def _on_diag(self, msg):
        kv = {}
        if msg.status:
            for v in msg.status[0].values:
                kv[v.key] = v.value
        self.diag_msgs.append(kv)

if not rclpy.ok():
    rclpy.init()
node = Sniffer()

deadline = time.time() + COLLECT_SECONDS
while time.time() < deadline:
    rclpy.spin_once(node, timeout_sec=0.1)

print(f'/joint_states     : {len(node.joint_msgs)} msgs')
print(f'/sim/diagnostics  : {len(node.diag_msgs)} msgs')
if not node.joint_msgs:
    print('\n[!] 메시지 없음 — sim 이 떠 있는지, ROS_DOMAIN_ID/RMW 가 일치하는지 확인.')

## 2. 관절 각도 시간추이 플롯

freerun 모드에서는 `/joint_states` 가 ~50–60 Hz 로 들어오고,
처음 메시지의 `name` 순서를 그대로 사용합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

assert node.joint_msgs, 'joint_states 메시지가 비어 있어 그릴 수 없음'

names = node.joint_msgs[0][1]
ts = np.array([m[0] for m in node.joint_msgs])
ts -= ts[0]
pos = np.array([m[2] for m in node.joint_msgs])  # (T, J)

fig, ax = plt.subplots(figsize=(9, 4))
for j, n in enumerate(names):
    ax.plot(ts, pos[:, j], label=n, linewidth=1.0)
ax.set_xlabel('time since first sample [s]')
ax.set_ylabel('joint position [rad]')
ax.set_title(f'/joint_states — {len(names)} joints over {ts[-1]:.2f}s')
ax.legend(fontsize=7, ncol=2, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. `/sim/diagnostics` — 1Hz 텔레메트리

브리지가 1Hz 로 publish 하는 키들 (`sim_time`, `step_hz`, `cmd_hz`,
`render_hz`, `state` 등). 마지막 스냅샷만 표로 출력합니다.

In [ ]:
if node.diag_msgs:
    last = node.diag_msgs[-1]
    width = max(len(k) for k in last) if last else 0
    for k in sorted(last):
        print(f'{k:<{width}}  {last[k]}')
else:
    print('/sim/diagnostics 메시지 없음 (1Hz 라 5초 안에 0~5개)')

## 4. 정리

노드/컨텍스트는 같은 노트북 안에서 다시 init 하지 않도록 명시적으로
내려둡니다. (셀 재실행 시 `rclpy.init()` 가 두 번 불리면 에러)

In [ ]:
node.destroy_node()
rclpy.shutdown()
print('shutdown OK')